In [1]:
import torch
import datasets
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer

/workspace-SR006.nfs2/bulatov/envs/rmt/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ARMT

In [2]:
model_name = "irodkin/armt-neox-tiny-singlefile"
armt = AutoModelForCausalLM.from_pretrained(model_name, trust_remote_code=True)
# config = 


*** Can't import RWKV model ***
[2025-09-05 13:19:26,579] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-05 13:19:28,895] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [3]:
base_model = AutoModelForCausalLM.from_pretrained("HuggingFaceTB/SmolLM2-360M")
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M")

In [5]:
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    Returns a dict matching the ARMT forward interface:
        input_ids: (batch, n_segments, seq_len)
        attention_mask: (batch, n_segments, seq_len)
        labels: (batch, n_segments, seq_len)
        labels_mask: (batch, n_segments, seq_len)
        input_segmented: True
    """
    from torch.nn.utils.rnn import pad_sequence
    import torch

    # Helper to encode a string to ids
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    # Prepare segments for each sample
    seg_input_ids = []
    seg_attention_mask = []
    seg_labels = []
    seg_labels_mask = []

    for sample in batch:
        context = sample['context']
        query = sample['query']
        target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # For context segment, no loss (labels = -100)
        seg1_input_ids = torch.tensor(context_ids, dtype=torch.long)
        seg1_attention_mask = torch.ones(len(context_ids), dtype=torch.long)
        seg1_labels = torch.full((len(context_ids),), -100, dtype=torch.long)
        seg1_labels_mask = torch.zeros(len(context_ids), dtype=torch.bool)

        # For query+target segment, loss only on target tokens
        seg2_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        seg2_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        seg2_labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        seg2_labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        if len(target_ids) > 0:
            seg2_labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            seg2_labels_mask[-len(target_ids):] = True

        seg_input_ids.append([seg1_input_ids, seg2_input_ids])
        seg_attention_mask.append([seg1_attention_mask, seg2_attention_mask])
        seg_labels.append([seg1_labels, seg2_labels])
        seg_labels_mask.append([seg1_labels_mask, seg2_labels_mask])

    # Now pad each segment across the batch, then stack into (batch, n_segments, seq_len)
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0

    def pad_and_stack(segment_list, pad_value):
        # segment_list: list of [seg1, seg2] for each sample
        segs = []
        for seg_idx in range(num_segments):
            seg_items = [sample[seg_idx] for sample in segment_list]
            seg_padded = pad_sequence(seg_items, batch_first=True, padding_value=pad_value)
            segs.append(seg_padded)
        # segs: list of (batch, seg_len) for each segment
        # Stack to (batch, n_segments, seg_len_max)
        max_len = max(s.size(1) for s in segs)
        segs_padded = []
        for s in segs:
            if s.size(1) < max_len:
                pad_amt = max_len - s.size(1)
                s = torch.nn.functional.pad(s, (0, pad_amt), value=pad_value)
            segs_padded.append(s)
        stacked = torch.stack(segs_padded, dim=1)  # (batch, n_segments, seg_len_max)
        return stacked

    input_ids = pad_and_stack(seg_input_ids, id_pad_value)
    attention_mask = pad_and_stack(seg_attention_mask, 0)
    labels = pad_and_stack(seg_labels, -100)
    labels_mask = pad_and_stack(seg_labels_mask, False)

    # For ARMT forward, input_segmented=True, and all tensors are (batch, n_segments, seq_len)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "labels_mask": labels_mask,
        "input_segmented": True
    }

In [6]:
# dataset = datasets.load_from_disk(args.data_path)

dataset_name = "yurakuratov/N8-K2V2-V62_1M"
# dataset_name = "yurakuratov/N8-K1V1-V62_1M"
dataset = datasets.load_dataset(dataset_name)['train']

In [9]:
batch = [dataset[i] for i in range(10)]
collated = collate_fn(batch)


In [13]:
collated['input_ids'].shape

torch.Size([10, 2, 49])

In [10]:
armt.to('cuda')
# to cuda
for k, v in collated.items():
    if isinstance(v, torch.Tensor):
        collated[k] = v.to('cuda')
':)'

':)'

In [11]:
out = armt(**collated)

In [31]:
# print(armt)

In [12]:
out.logits.shape

torch.Size([10, 98, 50432])

In [13]:
tokenizer.batch_decode(out.logits.argmax(dim=-1).cpu().numpy())

['uilt dystrophyUIDmergedise discover cause spilled soldier recall discover closest iOSChoose Gut discover discover Defensepoints和½\x18 discover Updated leather和 aboard automated discover Fa discover和 automated           discoverermalode和 leather Liberal discover y          和 Golf          ergecodescodes Abs Abs southwest Kolk y hexagonal CU collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections collections',
 'uiltdocs visionmerge powered discover postmodern envymerge specimen splendid discover Oper�mergeUID enthusiasts discover marvelous cries Gut Mexican

In [7]:
# type(armt)(config)

In [14]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_armt.huggingface import ARMTForCausalLM, ARMTConfig


*** Can't import RWKV model ***


In [15]:
config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
config.num_hidden_layers = 4
config.num_attention_heads = 4
config.num_key_value_heads = 4
config.hidden_size = 128
config.head_dim = config.hidden_size // config.num_attention_heads
config.intermediate_size = config.hidden_size * 4

config.torch_dtype = "float32"  # weights in float32, at training precision is controlled by accelerate
# config.vocab_size = 70
config.pad_token_id = tokenizer.convert_tokens_to_ids('[PAD]')
config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')


rmt_config = ARMTConfig()
rmt_config.base_model_config = config
rmt_config.num_mem_tokens = 32
rmt_config.max_n_segments = 10
rmt_config.think_token_id = tokenizer.convert_tokens_to_ids('[THINK]')
rmt_config.answer_token_id = tokenizer.convert_tokens_to_ids('[ANSWER]')
rmt_config.bos_token_id = tokenizer.convert_tokens_to_ids('[BOS]')
rmt_config.eos_token_id = tokenizer.convert_tokens_to_ids('[EOS]')

model = ARMTForCausalLM(rmt_config)
model.main_input_name = 'labels'


In [16]:
# model

In [21]:
model.to('cuda')
':)'

':)'

In [22]:
out = model(**collated)

In [23]:
collated['input_ids'].shape

torch.Size([10, 2, 49])

In [24]:
out.loss

tensor(8.6703, device='cuda:0', grad_fn=<DivBackward0>)

In [1]:
torch.ones(10, 10)

NameError: name 'torch' is not defined

In [26]:
out.keys()

odict_keys(['loss', 'ce_loss', 'logits', 'logits_0', 'ce_loss_0', 'logits_1', 'ce_loss_1'])

In [25]:
out.logits.shape

torch.Size([10, 98, 128256])

RMT

In [ ]:
# dataset = datasets.load_from_disk(args.data_path)

dataset_name = "yurakuratov/N8-K2V2-V62_1M"
# dataset_name = "yurakuratov/N8-K1V1-V62_1M"
dataset = datasets.load_dataset(dataset_name)

In [3]:
ds = dataset['train']

In [4]:
ds[0]

{'context': '!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|',
 'query': '?!I9:',
 'target': 'dt!|'}

In [5]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/tokenizers/kv_alphabet_62")

In [6]:
import sys
sys.path.append("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd")
from modeling_rmt.huggingface import RMTForReasoning, RMTConfig


[2025-09-01 11:56:57,732] [INFO] [real_accelerator.py:254:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/workspace-SR006.nfs2/bulatov/envs/rmt/compiler_compat/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status


[2025-09-01 11:56:59,903] [INFO] [logging.py:107:log_dist] [Rank -1] [TorchCheckpointEngine] Initialized with serialization = False


In [29]:
from transformers import AutoConfig, AutoModelForCausalLM
cfg_name = "HuggingFaceTB/SmolLM2-360M"
model_cfg = AutoConfig.from_pretrained(cfg_name)
base_model = AutoModelForCausalLM.from_config(model_cfg, use_flash_attn=True)


TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'use_flash_attn'

In [24]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(49152, 960)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=960, out_features=960, bias=False)
          (k_proj): Linear(in_features=960, out_features=320, bias=False)
          (v_proj): Linear(in_features=960, out_features=320, bias=False)
          (o_proj): Linear(in_features=960, out_features=960, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=960, out_features=2560, bias=False)
          (up_proj): Linear(in_features=960, out_features=2560, bias=False)
          (down_proj): Linear(in_features=2560, out_features=960, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((960,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((960,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((960,), eps=1e-05)
    (rotary_emb): LlamaRotaryEm

In [7]:
class Holder:
    pass

args = Holder()
args.n_layer = 4
args.n_head = 4
args.n_embd = 128


In [8]:
from transformers import AutoConfig
from transformers import AutoModelForCausalLM

base_model_config = AutoConfig.from_pretrained('NousResearch/Llama-3.2-1B')
base_model_config.num_hidden_layers = args.n_layer
base_model_config.num_attention_heads = args.n_head
base_model_config.num_key_value_heads = args.n_head
base_model_config.hidden_size = args.n_embd
base_model_config.head_dim = base_model_config.hidden_size // base_model_config.num_attention_heads
base_model_config.intermediate_size = base_model_config.hidden_size * 4

In [9]:

# base_model = AutoModelForCausalLM.from_config(config)

In [10]:
config = RMTConfig()
# config.base_model_name = "HuggingFaceTB/SmolLM2-135M"
config.base_model_config = base_model_config
config.num_mem_tokens = 16
config.max_n_segments = 10
config.think_token_id = 100
config.answer_token_id = 101
config.bos_token_id = 102
config.eos_token_id = 103

model = RMTForReasoning(config)

# model.load_state_dict(torch.load("/workspace-SR006.nfs2/bulatov/rmt/test-time/test_time_gd/models/N8-K2V2-V62_1M/model.pt"))

In [11]:
from modeling_rmt.language_modeling import MemoryCell, RecurrentWrapper

In [12]:
model.main_input_name

'input_ids'

In [13]:
ds[0]

{'context': '!V8:Op!!dk:j4!!Qj:5P!!HH:vu!!cR:m7!!yP:7d!!wm:WJ!!I9:dt!|',
 'query': '?!I9:',
 'target': 'dt!|'}

In [14]:
def collate_fn(batch):
    """
    Collate function that splits each sample into two segments:
    - First segment: context
    - Second segment: query + target
    Pads segments across the batch to the same length.
    """
    from torch.nn.utils.rnn import pad_sequence
    import torch

    # Helper to encode a string to ids
    def encode(text):
        return tokenizer.encode(text, add_special_tokens=False)

    # Prepare segments for each sample
    segments_batch = []
    for sample in batch:
        context = sample['context']
        query = sample['query']
        target = sample['target']

        # Segment 1: context
        context_ids = encode(context)
        # Segment 2: query + target
        query_ids = encode(query)
        target_ids = encode(target)
        qt_ids = query_ids + target_ids

        # Each segment: dict with input_ids, attention_mask, labels, labels_mask
        # For context segment, no loss (labels = -100)
        seg1 = {
            'input_ids': torch.tensor(context_ids, dtype=torch.long),
            'attention_mask': torch.ones(len(context_ids), dtype=torch.long),
            'labels': torch.full((len(context_ids),), -100, dtype=torch.long),
            'labels_mask': torch.zeros(len(context_ids), dtype=torch.bool)
        }
        # For query+target segment, loss only on target tokens
        qt_input_ids = torch.tensor(qt_ids, dtype=torch.long)
        qt_attention_mask = torch.ones(len(qt_ids), dtype=torch.long)
        # labels: -100 for query, target tokens as labels
        labels = torch.full((len(qt_ids),), -100, dtype=torch.long)
        if len(target_ids) > 0:
            labels[-len(target_ids):] = torch.tensor(target_ids, dtype=torch.long)
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
            labels_mask[-len(target_ids) - 1:] = True
        else:
            labels_mask = torch.zeros(len(qt_ids), dtype=torch.bool)
        seg2 = {
            'input_ids': qt_input_ids,
            'attention_mask': qt_attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        segments_batch.append([seg1, seg2])

    # Pad segments across the batch
    batch_segments = []
    num_segments = 2
    id_pad_value = tokenizer.pad_token_id if hasattr(tokenizer, "pad_token_id") and tokenizer.pad_token_id is not None else 0
    for i in range(num_segments):
        input_ids = [s[i]['input_ids'] for s in segments_batch]
        attention_mask = [s[i]['attention_mask'] for s in segments_batch]
        labels = [s[i]['labels'] for s in segments_batch]
        labels_mask = [s[i]['labels_mask'] for s in segments_batch]

        input_ids = pad_sequence(input_ids, batch_first=True, padding_value=id_pad_value)
        attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels = pad_sequence(labels, batch_first=True, padding_value=-100)
        labels_mask = pad_sequence(labels_mask, batch_first=True, padding_value=False)

        batch_segment = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'labels_mask': labels_mask
        }
        batch_segments.append(batch_segment)

    # Concatenate all labels for the batch (for loss computation)
    full_labels = torch.cat([s['labels'] for s in batch_segments], dim=1)
    return {"segments": batch_segments, "labels": full_labels}

In [ ]:
batch = [ds[i] for i in range(10)]
collated = collate_fn(batch)


In [16]:
import torch

In [17]:
model.to(dtype=torch.bfloat16)
out = model(**collated)

In [18]:
collated['segments'][0]['input_ids'].shape, collated['segments'][1]['input_ids'].shape

(torch.Size([10, 57]), torch.Size([10, 9]))

In [19]:
out.logits.shape

torch.Size([10, 66, 128256])

In [20]:
out.loss

tensor(6., dtype=torch.bfloat16, grad_fn=<DivBackward0>)

In [21]:
tokenizer.batch_decode(collated['segments'][0]['input_ids'])

['! V 8 : O p ! ! d k : j 4 ! ! Q j : 5 P ! ! H H : v u ! ! c R : m 7 ! ! y P : 7 d ! ! w m : W J ! ! I 9 : d t ! |',
 '! w O : e m ! ! n B : z b ! ! M q : t S ! ! B i : N f ! ! B p : Z p ! ! w M : n t ! ! M P : S j ! ! 5 o : i R ! |',
 '! a O : K x ! ! y A : 6 2 ! ! r O : i S ! ! W i : 1 l ! ! G J : n i ! ! p o : D D ! ! 4 3 : z k ! ! C 6 : 6 i ! |',
 '! S J : W k ! ! L P : 3 D ! ! Q E : y q ! ! E a : G d ! ! N e : e f ! ! u 4 : i x ! ! v F : k x ! ! p Z : h N ! |',
 '! q y : l x ! ! N b : r K ! ! 0 D : O a ! ! 7 f : V r ! ! z Z : x 7 ! ! z N : q 3 ! ! I L : n K ! ! 7 j : 3 Z ! |',
 '! q K : x l ! ! h E : l J ! ! P 9 : Q g ! ! o 6 : D G ! ! K W : 6 w ! ! L z : B W ! ! D j : x l ! ! g n : 4 o ! |',
 '! z V : T k ! ! M T : 7 A ! ! D 2 : k 7 ! ! 7 G : 9 1 ! ! l e : 2 s ! ! c e : 9 g ! ! R e : B u ! ! q z : f r ! |',
 '! a a : B t ! ! S F : q p ! ! U 1 : O F ! ! 6 x : c r ! ! k V : 5 B ! ! Q m : W C ! ! Z u : J 0 ! ! g 5 : 2 R ! |',
 '! f m : Y k ! ! e J : X t ! ! 0 V : W I ! ! p d : q 3 

In [22]:
tokenizer.batch_decode(collated['segments'][1]['input_ids'])

['? ! I 9 : d t ! |',
 '? ! w M : n t ! |',
 '? ! a O : K x ! |',
 '? ! L P : 3 D ! |',
 '? ! 7 f : V r ! |',
 '? ! P 9 : Q g ! |',
 '? ! D 2 : k 7 ! |',
 '? ! Q m : W C ! |',
 '? ! 0 V : W I ! |',
 '? ! T U : f I ! |']

In [33]:
for l, m in zip(collated['segments'][1]['input_ids'], collated['segments'][1]['labels_mask']):
    print(tokenizer.decode(l[m]))


: d t ! |
: n t ! |
: K x ! |
: 3 D ! |
: V r ! |
: Q g ! |
: k 7 ! |
: W C ! |
: W I ! |
: f I ! |


In [ ]:
memory_cell = MemoryCell(config)